In [ ]:
# =============================================================================
# COMPLETE END-TO-END SLEEP STAGING PIPELINE: HIERARCHICAL SleepGNN
# 14-Node Graph (Full HRV & Kinematics) + BiGRU Sequence Modeling + 5-Fold CV
# =============================================================================

import os
import io
import zipfile
import copy
import random
import scipy.io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from scipy.interpolate import interp1d
from collections import OrderedDict
from tqdm.auto import tqdm, trange

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score, classification_report

# PyTorch Geometric Imports
try:
    import torch_geometric.nn as pyg_nn
    from torch_geometric.data import Batch
except ImportError:
    print("Please install PyTorch Geometric: pip install torch_geometric")

# --------------------------- 1. CONFIGURATION & SEEDING ---------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

LOCAL_DATA_DIR = "./dataset/a-multi-night-instantaneous-heart-rate-and-accelerometry-dataset-with-eeg-sleep-stage-labels-1.0.0"
DATASET_FOLDER_NAME = "a-multi-night-instantaneous-heart-rate-and-accelerometry-dataset-with-eeg-sleep-stage-labels-1.0.0"

# Hyperparameters
USE_SUBJECT_LIMIT = None  
NUM_WORKERS = 2           
NIGHT_CACHE_SIZE = 3
BATCH_SIZE = 64           # Smaller batch size because we load sequences of epochs (T=20)
SEQ_WINDOW_LEN = 20       # Number of continuous epochs fed into BiGRU at once
EPOCHS = 50
LEARNING_RATE = 1e-3
EPOCH_WINDOW_SEC = 30
INTERPOLATION_FREQ_HZ = 10
SEQ_LEN = EPOCH_WINDOW_SEC * INTERPOLATION_FREQ_HZ # 300 feature points per node

# Locate Dataset
def _find_kaggle_zip():
    patterns = ["/kaggle/input/datasets/jbjoykumar/a-mult1-zip-6-35-gb/A-MULT1.ZIP", "/kaggle/input/**/A-MULT1.ZIP", "/kaggle/input/**/*.zip"]
    for pattern in patterns:
        hits = glob(pattern, recursive=True)
        if hits: return hits[0]
    return None

ZIP_PATH = _find_kaggle_zip()
DATA_DIR = LOCAL_DATA_DIR if os.path.exists(LOCAL_DATA_DIR) else None
DATA_MODE = "zip" if ZIP_PATH else ("filesystem" if DATA_DIR else None)

# --------------------------- 2. GNN SIGNAL PROCESSING (14 FEATURES) ---------------------------
def robust_scale(arr):
    median = np.nanmedian(arr)
    iqr = np.nanpercentile(arr, 75) - np.nanpercentile(arr, 25)
    return (arr - median) / (iqr + 1e-6) if iqr > 0 else arr - median

def process_night_signals_gnn_full(hr_df, mot_df, labels):
    """
    Constructs the 14 specific nodes required for the Graph.
    Preserves all HRV, Kinematic, and Contextual features.
    """
    hr_df = hr_df.apply(pd.to_numeric, errors='coerce').dropna()
    mot_df = mot_df.apply(pd.to_numeric, errors='coerce').dropna()
    if hr_df.empty or mot_df.empty: return None, None
    
    # Clip extreme artifacts safely
    for col in ['hr']:
        mu, sigma = hr_df[col].mean(), hr_df[col].std()
        if sigma > 0: hr_df[col] = np.clip(hr_df[col], mu - 3*sigma, mu + 3*sigma)
    for col in ['x', 'y', 'z']:
        mu, sigma = mot_df[col].mean(), mot_df[col].std()
        if sigma > 0: mot_df[col] = np.clip(mot_df[col], mu - 3*sigma, mu + 3*sigma)
        
    start_time = max(hr_df['time'].min(), mot_df['time'].min())
    hr_df = hr_df.copy(); mot_df = mot_df.copy()
    hr_df['time'] -= start_time; mot_df['time'] -= start_time
    hr_df = hr_df[hr_df['time'] >= 0].reset_index(drop=True)
    mot_df = mot_df[mot_df['time'] >= 0].reset_index(drop=True)
    
    # Sampling Frequencies
    time_diff = np.diff(hr_df['time'].values); time_diff[time_diff == 0] = 1e-6
    hr_df['freq'] = np.insert(1.0 / time_diff, 0, 1.0 / time_diff[0])
    mot_df['mag'] = np.sqrt(mot_df['x']**2 + mot_df['y']**2 + mot_df['z']**2)
    
    # Label alignment
    label_map = {0:0, 1:1, 2:2, 3:3, 5:4}
    labels = np.array([label_map.get(int(l), int(l)) for l in labels])
    target_time = np.arange(0, len(labels) * EPOCH_WINDOW_SEC, 1.0 / INTERPOLATION_FREQ_HZ)
    
    # Base Interpolations
    hr_resampled = np.clip(interp1d(hr_df['time'].values, hr_df['hr'].values, fill_value='extrapolate')(target_time), hr_df['hr'].min(), hr_df['hr'].max())
    freq_resampled = np.clip(interp1d(hr_df['time'].values, hr_df['freq'].values, fill_value='extrapolate')(target_time), hr_df['freq'].min(), hr_df['freq'].max())
    x_resampled = np.clip(interp1d(mot_df['time'].values, mot_df['x'].values, fill_value='extrapolate')(target_time), mot_df['x'].min(), mot_df['x'].max())
    y_resampled = np.clip(interp1d(mot_df['time'].values, mot_df['y'].values, fill_value='extrapolate')(target_time), mot_df['y'].min(), mot_df['y'].max())
    z_resampled = np.clip(interp1d(mot_df['time'].values, mot_df['z'].values, fill_value='extrapolate')(target_time), mot_df['z'].min(), mot_df['z'].max())
    mag_resampled = np.clip(interp1d(mot_df['time'].values, mot_df['mag'].values, fill_value='extrapolate')(target_time), mot_df['mag'].min(), mot_df['mag'].max())
    
    # --- Advanced Cardiac Features (HRV) ---
    hr_clipped = np.clip(hr_resampled, 30, 200) 
    rr_intervals = 60000.0 / hr_clipped 
    rr_series = pd.Series(rr_intervals)
    
    window = INTERPOLATION_FREQ_HZ * 30 
    sdnn = rr_series.rolling(window, min_periods=1, center=False).std().fillna(0).values
    rr_diff = np.insert(np.diff(rr_intervals), 0, 0)
    rmssd = np.sqrt(pd.Series(rr_diff**2).rolling(window, min_periods=1, center=False).mean().fillna(0).values)
    pnn50 = pd.Series((np.abs(rr_diff) > 50).astype(float)).rolling(window, min_periods=1, center=False).mean().fillna(0).values
    
    # --- Advanced Motion Features (Kinematics) ---
    jerk = np.insert(np.diff(mag_resampled), 0, 0)
    energy = pd.Series(mag_resampled**2).rolling(window, min_periods=1, center=False).mean().fillna(0).values
    act_count = pd.Series((mag_resampled > np.percentile(mag_resampled, 75)).astype(float)).rolling(window, min_periods=1, center=False).sum().fillna(0).values

    # --- Contextual Time ---
    progress = target_time / target_time[-1] if len(target_time) > 1 else np.zeros_like(target_time)
    time_cos = np.cos(2 * np.pi * progress)

    # 14-Node Graph Stack
    channels = [
        hr_resampled, rr_intervals, rmssd, sdnn, pnn50, freq_resampled,   # Nodes 0-5
        x_resampled, y_resampled, z_resampled, mag_resampled, jerk, energy, act_count, # Nodes 6-12
        time_cos # Node 13
    ]
    
    full_signals = np.vstack([robust_scale(c) for c in channels])
    
    max_epochs = min(len(labels), full_signals.shape[1] // SEQ_LEN)
    if max_epochs == 0: return None, None
    epochs_data = [full_signals[:, i*SEQ_LEN:(i+1)*SEQ_LEN] for i in range(max_epochs)]
    return np.array(epochs_data, dtype=np.float32), labels[:max_epochs].astype(np.int64)

# --------------------------- 3. DATA LOADERS & SEQUENCING ---------------------------
class ZipNightSource:
    def __init__(self, zip_path, subject_limit=None):
        self.zip_path = zip_path; self.subject_limit = subject_limit
        self._night_paths = self._index_zip()
        
    def _index_zip(self):
        with zipfile.ZipFile(self.zip_path, 'r') as zf:
            hr_files = [n for n in zf.namelist() if n.lower().endswith('/hr.csv') or n.lower().endswith('hr.csv')]
        night_map = {}
        for hr_name in hr_files:
            parts = [p for p in hr_name.replace('\\','/').split('/') if p]
            if len(parts) < 3: continue
            subject_id, night_id = parts[-3], parts[-2]
            night_map[(subject_id, night_id)] = '/'.join(parts[:-3])
        nights = sorted(night_map.keys(), key=lambda x: (x[0], int(x[1]) if x[1].isdigit() else x[1]))
        if self.subject_limit:
            allowed = sorted({s for s, _ in nights})[:self.subject_limit]
            nights = [(s, n) for s, n in nights if s in allowed]
        self._prefix_map = night_map; return nights
        
    def list_nights(self): return list(self._night_paths)
    
    def load_night(self, subject_id, night_id):
        prefix = self._prefix_map.get((subject_id, str(night_id)))
        base = f"{prefix}/{subject_id}/{night_id}/" if prefix else f"{subject_id}/{night_id}/"
        hr_n, mot_n, lab_n = base+'hr.csv', base+'motion.csv', base+'labels.mat'
        try:
            with zipfile.ZipFile(self.zip_path, 'r') as zf:
                hr_df = pd.read_csv(io.BytesIO(zf.read(hr_n)), header=None, names=['time','hr'], dtype=str)
                mot_df = pd.read_csv(io.BytesIO(zf.read(mot_n)), header=None, names=['time','x','y','z'], dtype=str)
                mat_data = scipy.io.loadmat(io.BytesIO(zf.read(lab_n)))
                k = [k for k in mat_data.keys() if any(x in k.lower() for x in ('label','stage','hypno'))][0]
                labels = mat_data[k].flatten()
        except: return None, None
        return process_night_signals_gnn_full(hr_df, mot_df, labels)

DATA_SOURCE = ZipNightSource(ZIP_PATH, USE_SUBJECT_LIMIT) if DATA_MODE == 'zip' else None # Fallbacks omitted for brevity if Zip found

class LazyGNNWindowDataset(Dataset):
    """Yields Temporal Sequences of Epochs (T_seq, Nodes=14, Features=300)"""
    def __init__(self, night_list, source=None, seq_len=SEQ_WINDOW_LEN, cache_size=NIGHT_CACHE_SIZE):
        self.nights, self.source, self.seq_len = list(night_list), source or DATA_SOURCE, seq_len
        self.cache_size = cache_size
        self._cache, self.windows = OrderedDict(), []
        for night_idx, (s_id, n_id) in enumerate(tqdm(self.nights, desc='Indexing sequences', leave=False)):
            X, _ = self._load_night(night_idx)
            if X is None or len(X) < seq_len: continue
            for pos in range(0, len(X) - seq_len + 1, seq_len // 2): 
                self.windows.append((night_idx, pos))
                
    def _load_night(self, night_idx):
        key = self.nights[night_idx]
        if key not in self._cache:
            X, y = self.source.load_night(key[0], key[1])
            if X is None: return None, None
            self._cache[key] = (X, y); self._cache.move_to_end(key)
            while len(self._cache) > self.cache_size: self._cache.popitem(last=False)
        return self._cache[key]
    
    def __len__(self): return len(self.windows)
    
    def __getitem__(self, idx):
        night_idx, pos = self.windows[idx]
        X, y = self._load_night(night_idx)
        X_seq = X[pos : pos + self.seq_len].astype(np.float32)
        y_seq = y[pos : pos + self.seq_len].astype(np.int64)
        return torch.from_numpy(X_seq), torch.from_numpy(y_seq)

# --------------------------- 4. PYG GRAPH CONVERSION UTILITY (14 NODES) ---------------------------
def tensor_to_pyg_batch(sensor_tensor: torch.Tensor):
    """
    On-the-fly conversion of 4D batch tensors to PyG Batch.
    sensor_tensor shape: (B, T, N=14, F=300)
    """
    B, T, N, F_dim = sensor_tensor.shape
    num_graphs = B * T
    
    # 1. Flatten into (Total Nodes, Features)
    x = sensor_tensor.view(num_graphs * N, F_dim)
    
    # 2. Build Fully Connected Edge Index for 1 Graph (14x14)
    base_edge_index = torch.cartesian_prod(torch.arange(N), torch.arange(N)).t().contiguous()
    
    # 3. Vectorize across the entire batch/sequence
    offset = torch.arange(0, num_graphs * N, step=N, device=sensor_tensor.device).view(-1, 1)
    edge_index = (base_edge_index.unsqueeze(0).to(sensor_tensor.device) + offset).view(2, -1)
    
    # 4. Batch mapping vector for global pooling
    batch_idx = torch.arange(num_graphs, device=sensor_tensor.device).repeat_interleave(N)
    
    return Batch(x=x, edge_index=edge_index, batch=batch_idx), B, T

# --------------------------- 5. HIERARCHICAL SleepGNN ARCHITECTURE ---------------------------
class HierarchicalSleepGNN(nn.Module):
    def __init__(self, in_feats=300, gat_hidden=64, gat_heads=4, gru_hidden=128, num_classes=5, dropout=0.3):
        super(HierarchicalSleepGNN, self).__init__()
        
        # --- SPATIAL BLOCK (GAT) ---
        # GATConv computes attention across all 14 nodes dynamically
        self.gat1 = pyg_nn.GATConv(in_channels=in_feats, out_channels=gat_hidden, heads=gat_heads, concat=True) 
        self.gat2 = pyg_nn.GATConv(in_channels=gat_hidden * gat_heads, out_channels=gat_hidden, heads=1, concat=False)
        
        # --- TEMPORAL BLOCK (BiGRU) ---
        self.gru = nn.GRU(input_size=gat_hidden, hidden_size=gru_hidden, num_layers=2, 
                          batch_first=True, bidirectional=True, dropout=dropout)
        
        # --- CLASSIFICATION HEAD ---
        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden * 2, gru_hidden),
            nn.LayerNorm(gru_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden, num_classes)
        )

    def forward(self, x_tensor):
        """x_tensor: (B, T, N=14, F=300)"""
        # 1. Convert to PyG Graph Batch
        pyg_batch, B, T = tensor_to_pyg_batch(x_tensor)
        x, edge_index, batch_idx = pyg_batch.x, pyg_batch.edge_index, pyg_batch.batch
        
        # 2. Spatial Message Passing (14 Nodes exchanging data)
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        
        # 3. Graph Pooling -> Collapse 14 nodes into 1 Epoch Embedding
        pooled_epochs = pyg_nn.global_max_pool(x, batch_idx)  # (B*T, gat_hidden)
        
        # 4. Temporal Sequence Modeling
        seq_epochs = pooled_epochs.view(B, T, -1)
        gru_out, _ = self.gru(seq_epochs)  # (B, T, gru_hidden * 2)
        
        # 5. Classify sequence of epochs
        return self.classifier(gru_out)    # (B, T, num_classes)

class FocalLoss(nn.Module):
    """Handles class imbalance explicitly."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        return (((1 - torch.exp(-ce_loss)) ** self.gamma) * ce_loss).mean()

# --------------------------- 6. TRAINING & 5-FOLD CV ENGINE ---------------------------
def run_5fold_cv_gnn():
    if not DATA_SOURCE: return
    
    print("Discovering nights...")
    all_nights = DATA_SOURCE.list_nights()
    unique_subjects = sorted({s for s, _ in all_nights})
    subject_groups = np.array(unique_subjects)
    gkf = GroupKFold(n_splits=5)
    
    fold_metrics = {'accuracy': [], 'kappa': [], 'macro_f1': []}
    
    for fold, (t_idx, v_idx) in enumerate(gkf.split(X=subject_groups, y=None, groups=subject_groups)):
        print("="*80); print(f"🚀 FOLD {fold+1}/5 - 14-Node Hierarchical SleepGNN"); print("="*80)
        
        t_subj, v_subj = set(subject_groups[t_idx]), set(subject_groups[v_idx])
        t_n = [(s, n) for s, n in all_nights if s in t_subj]
        v_n = [(s, n) for s, n in all_nights if s in v_subj]
        
        train_dataset = LazyGNNWindowDataset(t_n)
        test_dataset = LazyGNNWindowDataset(v_n)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
        
        # Calculate Class Weights
        y_train_labels = [] 
        for s_id, n_id in tqdm(t_n, desc=f'Fold {fold+1} Label Scan', leave=False):
            _, y = DATA_SOURCE.load_night(s_id, n_id)
            if y is not None: y_train_labels.extend(y.tolist())
        weights = compute_class_weight('balanced', classes=np.arange(5), y=np.array(y_train_labels, dtype=np.int64))
        weight_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
        
        # Initialize 14-Node Model
        model = HierarchicalSleepGNN(in_feats=SEQ_LEN, num_classes=5).to(device)
        criterion = FocalLoss(alpha=weight_tensor, gamma=2.0)
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
        scaler = torch.cuda.amp.GradScaler()
        
        best_kappa, patience = -1.0, 0
        
        for epoch in range(1, EPOCHS + 1):
            model.train()
            train_loss = 0.0
            
            for xb, yb in tqdm(train_loader, desc=f'Train E{epoch}', leave=False):
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                
                with torch.cuda.amp.autocast(): 
                    out = model(xb) # Shape: (B, T, 5)
                    loss = criterion(out.view(-1, 5), yb.view(-1)) 
                    
                if not torch.isfinite(loss): continue
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                train_loss += loss.item() * xb.size(0)
                
            scheduler.step()
            
            # Validation
            model.eval()
            val_preds, val_targets, val_loss = [], [], 0.0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    out = model(xb)
                    loss = criterion(out.view(-1, 5), yb.view(-1))
                    val_loss += loss.item() * xb.size(0)
                    
                    val_preds.extend(out.argmax(dim=-1).view(-1).cpu().numpy())
                    val_targets.extend(yb.view(-1).cpu().numpy())
            
            kappa = cohen_kappa_score(val_targets, val_preds)
            f1 = f1_score(val_targets, val_preds, average='macro')
            
            print(f"Epoch {epoch} | T-Loss: {train_loss/max(1, len(train_dataset)):.4f} | V-Loss: {val_loss/max(1, len(test_dataset)):.4f} | Kappa: {kappa:.4f}")
            
            if kappa > best_kappa:
                best_kappa, patience = kappa, 0
                torch.save(model.state_dict(), f"sleepgnn_14node_fold_{fold+1}_best.pt")
            else:
                patience += 1
                if patience >= 10: 
                    print(f"Early stopping triggered. Best Kappa: {best_kappa:.4f}")
                    break
                    
        # Fold Results
        model.load_state_dict(torch.load(f"sleepgnn_14node_fold_{fold+1}_best.pt"))
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                val_preds.extend(model(xb.to(device)).argmax(dim=-1).view(-1).cpu().numpy())
                val_targets.extend(yb.view(-1).numpy())
                
        fold_metrics['accuracy'].append(accuracy_score(val_targets, val_preds))
        fold_metrics['kappa'].append(cohen_kappa_score(val_targets, val_preds))
        fold_metrics['macro_f1'].append(f1_score(val_targets, val_preds, average='macro'))
        
        print(f"--- Fold {fold+1} Test Results ---")
        print(classification_report(val_targets, val_preds, target_names=['Wake','N1','N2','N3','REM'], zero_division=0))
        
    print("="*80); print("FINAL 5-FOLD CROSS VALIDATION RESULTS (14-Node SleepGNN)"); print("="*80)
    for metric, vals in fold_metrics.items():
        print(f"{metric.capitalize()}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

if __name__ == '__main__':
    run_5fold_cv_gnn()